<img src='../OUTILS/bandeau_MF.png' align='right' width='100%'/>

<div class="alert alert-info alert-success">
<h3>Élaboration d'image composite MTG / FCI avec SATPY </h3></div>

## <a id='TOC-TOP'></a>Contenus

<div class="alert alert-block alert-warning">
    
<b>PREREQUIS </b>
    
Ce Notebook ne nécessite pas de prérequis. 
Le kernel "env_MF_teledetection" doit être utilisé

</div>
<hr>

SatPy

Il s'agit d'une bibliothèque Python pour le traitement des données des satellites météorologiques

Satpy permet de lire, manipuler et écrire des données issues d'instruments satellites météorologiques d'observation de la Terre.

Satpy fournit la possibilité de lire les données de différents formats (readers) et de manipuler les paramètres géophysiques de ces différents formats de fichiers.

Satpy fournit également des interfaces pour créer des images RVB (rouge/vert/bleu) qui combinent les données de plusieurs bandes d'instruments ou produits.

Diverses corrections atmosphériques et améliorations visuelles sont fournies pour améliorer le rendu et la qualité des images de sortie. Les données de sortie peuvent être écrites dans plusieurs formats de fichiers tels que les fichiers NetCDF, PNG et GeoTIFF.

Satpy permet également aux utilisateurs de rééchantillonner les données sur des grilles géographiques projetées (zones).

Pour obtenir de la documentation sur Satpy, quelques exemples, et un tutoriel de démarrage rapide : https://satpy.readthedocs.io/en/latest/index.html

<div class="alert alert-info" role="alert">

### <a id='section1'></a>1.Tout d'abord, il faut procéder à l'importation des librairies nécessaires.

</div>

In [ ]:
from satpy.scene import Scene
from satpy import find_files_and_readers
from datetime import datetime
import sys
from pyresample.geometry import AreaDefinition
import numpy as np
import os
from osgeo import gdal
from PIL import Image
import subprocess
from IPython.display import display,HTML
os.environ['PATH'] = f"/opt/conda/env_MF_teledetection/bin:{os.environ['PATH']}" 
os.environ['PATH'] = f"~/.conda/envs/env_MF_teledetection/bin:{os.environ['PATH']}"
os.environ['GDAL_DATA'] = '/opt/conda/env_MF_teledetection/share/gdal'
os.environ['PROJ_LIB'] = '/opt/conda/env_MF_teledetection/share/proj'

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>2.Définition du produit, date et heure, reader, et répertoires.

</div>

### Données d'entrée : fichiers chunk mtg (40) 
Pour télécharger d'autres fichiers, Cf. Jupyter Notebook  <a href="eumdac.ipynb" target="_blank"> Eumdac : téléchargement de données MTG</a></br>

In [ ]:
input = '/stockage/DATA/202509271200/'

In [ ]:
cd ~/MF_DATA_MANIPULATION

In [ ]:
download_dir = os.path.join(os.getcwd(), "RESULTS")
os.makedirs(download_dir, exist_ok=True)

In [ ]:
output='RESULTS'

In [ ]:
!ls $input

### SATPY permet de produire notamment des RGB, parmi les suivantes
airmass </br> ash </br>cimss_cloud_type</br> cloud_phase</br> cloudtop</br> convection</br> day_microphysics</br> dust</br> fog</br>  natural_color</br> night_fog</br> night_microphysics</br>  true_color</br> 

Définition du produit

In [ ]:
composite_name = "true_color"

### Définition de la date et heure de la composite à créer
(202401091200 disponible par défaut)

In [ ]:
annee='2025'
mois='09'
jour='27'
heure='12'

In [ ]:
yyyy=int(annee)
mm=int(mois)
dd=int(jour)
hh_debut=int(heure)
min_debut=int('00')
hh_fin=int(heure)
min_fin=int('10')

### Définition du reader

In [ ]:
reader_to_use = "fci_l1c_nc"

In [ ]:
filename = (output + '/RGB_satpy_' + composite_name )

In [ ]:
myfiles = find_files_and_readers(base_dir=input,
                                 start_time=datetime(yyyy,mm,dd,hh_debut,min_debut),
                                 end_time=datetime(yyyy,mm,dd,hh_fin,min_fin),
                                 reader=reader_to_use)

In [ ]:
scn = Scene(filenames=myfiles)

### Liste de toutes les composites disponibles

In [ ]:
print(scn.available_composite_names())

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>3.Création du produit.

</div>

In [ ]:
scn.load([composite_name], upper_right_corner='NE')

Ne pas tenir compte des avertissements

In [ ]:
natscn = scn.resample(scn.coarsest_area(), resampler='nearest')

In [ ]:
natscn.save_dataset(composite_name, filename=filename + '.tif')

La création du tif peut prendre quelques dizaines de secondes (~ 200 Mo pour un plein disque à 1 km soit 11136x11136 pixels)

In [ ]:
!identify -quiet {filename}.tif

### Défnition de la taille de l'image globe (pixels)

In [ ]:
!gdalwarp -overwrite -ts 1800 1800 {filename}.tif {filename}_redim.tif

In [ ]:
taille_redim=600  # si nécessaire pour affichage plus léger, 1800x1800 par défaut

In [ ]:
!convert  {filename}_redim.tif {filename}_redim_HD.jpg > /dev/null 2>&1
!convert -resize {taille_redim} {filename}_redim.tif {filename}_redim.jpg > /dev/null 2>&1

In [ ]:
imgredim = Image.open(filename + '_redim.jpg')
display(imgredim)

Découpe de la partie plein disque

In [ ]:
!convert -size 1800x1800 xc:black -fill white -draw "circle 900,900 900,29" mask.png

In [ ]:
!convert {filename}_redim_HD.jpg mask.png -alpha off -compose CopyOpacity -composite -alpha on {filename}_redim_HD.png

In [ ]:
imgredim_png = Image.open(filename + '_redim_HD.png')
display(imgredim_png)

Quelques liens utiles: </br>
<img src='../OUTILS/icon_pdf.png' alt='icon pdf' width='30px'></img><a href="../DOCS/Fiches_produits_MTG_legendes-RGB.pdf" target="_blank"> Principales légendes des produits RGB</a>


<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>4.Pour aller plus loin, essayons de comprendre l'image que nous voyons, en extrayant les 3 bandes de ce produits RGB

</div>

In [ ]:
!gdal_translate -q -b 1 {filename}_redim.tif {filename}_redim_bande_rouge.tif
!gdal_translate -q -b 2 {filename}_redim.tif {filename}_redim_bande_verte.tif
!gdal_translate -q -b 3 {filename}_redim.tif {filename}_redim_bande_bleue.tif
!convert -resize 300 {filename}_redim_bande_rouge.tif {filename}_redim_bande_rouge.jpg > /dev/null 2>&1
!convert -resize 300 {filename}_redim_bande_verte.tif {filename}_redim_bande_verte.jpg > /dev/null 2>&1
!convert -resize 300 {filename}_redim_bande_bleue.tif {filename}_redim_bande_bleue.jpg > /dev/null 2>&1

Affichage des 3 images RGB Rouge Verte Bleue

In [ ]:
imgrouge = Image.open(filename + '_redim_bande_rouge.jpg')
display(imgrouge)
imgverte = Image.open(filename + '_redim_bande_verte.jpg')
display(imgverte)
imgbleue = Image.open(filename + '_redim_bande_bleue.jpg')
display(imgbleue)

Pour l'exemple du produit Cloud Type RGB, les 3 bandes RGB sont celles respectivement associées aux canaux NIR1.38, VIS0.64 et NIR1.61 comme indiqué dans le document ci-dessous

### Document de synthèse des produits RGB FCI 

## &emsp;&emsp;&emsp;<a href="../DOCS/mtg_fci_synthese_produits.pdf" target="_blank"><img src='../DOCS/mtg_fci_synthese_produits.jpg' alt='tableau synthèse' width='500px'>  </a>

Autres exemples de visualisation d'un image composite rgb en 3 bandes rouge vert bleu : <a href="http://jupyter.fr-mf-general.s.ewcloud.host/user/coster/files/MF_DATA_MANIPULATION/DOCS/doc_mtg_fci_rgb_bande.html" title="lien "  > Ouvrir la page web de présentation ici</a>

### Ajout des frontières

In [ ]:
# Chemins vers les fichiers shapefile contenant les frontieres des pays : '../OUTILS/boundary/world-administrative-boundaries.shp' 

In [ ]:
#!gdal_rasterize -q -b 1 -burn 240 -b 2 -burn 240 -b 3 -burn 240 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp {filename}.tif >/dev/null 2>&1 # frontière blanche
!gdal_rasterize -q -b 1 -burn 255 -b 2 -burn 0 -b 3 -burn 0 -l world-administrative-boundaries OUTILS/boundary/world-administrative-boundaries.shp {filename}.tif >/dev/null 2>&1 # frontière rouge

### Affichage de l'image globe dans le jupyter notebook

Redimensionnement de l'image pour un affichage plus rapide

In [ ]:
!convert -resize 800 {filename}.tif {filename}_min.jpg > /dev/null 2>&1

In [ ]:
!ls -lrth {filename}.tif {filename}_min.jpg

In [ ]:
!identify {filename}.tif {filename}_min.jpg

In [ ]:
im = Image.open(filename + '_min.jpg')
display(im)

In [ ]:
!echo {filename}_min.jpg

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>5.Création d'une image à partir d'une zone de découpe 

</div>

In [ ]:
### Définition de la zone, et redimmensionnement de l'image 

In [ ]:
#exemple Espagne
coord_nord = 48
coord_sud = 33
coord_ouest = -20
coord_est = 5


In [ ]:
# Facteur de mise à l'échelle
scale_factor = 0.04  # 0.04 par défaut

In [ ]:
#Exemple de passage en système de coordonnées géographiques (GCS) système de coordonnées WGS84 afin d'utiliser des coordonnées en degrés ("EPSG:4326", similaire à cylindrique équidistant
#Définition de la zone de découpe (étendue géographique) -te (target extent) en degrés

In [ ]:
!gdalwarp -overwrite -t_srs "EPSG:4326" -te {coord_ouest} {coord_sud} {coord_est} {coord_nord}  {filename}.tif {filename}_decoupe_HD.tif
!gdalwarp -overwrite -t_srs "EPSG:4326" -te {coord_ouest} {coord_sud} {coord_est} {coord_nord}  {filename}.tif {filename}_decoupe_HD.png
!gdalwarp -overwrite -t_srs "EPSG:4326" -te {coord_ouest} {coord_sud} {coord_est} {coord_nord} {filename}.tif {filename}_decoupe_HD.jpg

In [ ]:
### Affichage des différentes tailles de fichiers

In [ ]:
!ls -lhtr {filename}_decoupe_HD.tif {filename}_decoupe_HD.png {filename}_decoupe_HD.jpg 

In [ ]:
!identify {filename}_decoupe_HD.tif {filename}_decoupe_HD.png {filename}_decoupe_HD.jpg 

In [ ]:
!convert -resize 1200 {filename}_decoupe_HD.png {filename}_decoupe_min.jpg
!ls -lhtr {filename}_decoupe_min.jpg

In [ ]:
im2 = Image.open(filename + '_decoupe_min.jpg')
display(im2)

<div class="alert alert-info" role="alert">
    
### <a id='section2'></a>6. Reprojection de l'image

</div>


#### Définition de la reprojection et de la zone de découpe de l'image à reprojeter

In [ ]:
#Exemple pour la projection orthographique ("ortho"):
#Modification des coordonnées du centre de l'image lat et lon
#Définition du redimmensionnent de l'image (facultatif) en spécifiant la largeur et la hauteur : option -ts (target size) en pixels
#Définition de la zone de découpe (étendue géographique) -te (target extent) en mètres
#Attention à conserver les proportions largeur/hauteur entre les options -ts et -te


In [ ]:
!gdalwarp -q -overwrite -t_srs "+proj=ortho +lat_0=41 +lon_0=-9" -ts 3200 1800 -te -1600000 -900000 1600000 900000 {filename}.tif {filename}_ortho.jpg

In [ ]:
!convert -resize 800 {filename}_ortho.jpg {filename}_ortho_min.png

In [ ]:
im3 = Image.open(filename + '_ortho_min.png')
display(im3)

### Nettoyage

In [ ]:
!rm {output}/*xml #{output}/*.tif

In [ ]:
#!rm -f {output}/*